In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('accepted_2007_to_2018Q4.csv')
df.info()
df.head()

C:\Users\alahd\AppData\Local\Temp\ipykernel_17360\2172562123.py:4: DtypeWarning: Columns (0: id, 1: desc, 2: next_pymnt_d, 3: verification_status_joint, 4: sec_app_earliest_cr_line, 5: hardship_type, 6: hardship_reason, 7: hardship_status, 8: hardship_start_date, 9: hardship_end_date, 10: payment_plan_start_date, 11: hardship_loan_status, 12: debt_settlement_flag_date, 13: settlement_status, 14: settlement_date) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('accepted_2007_to_2018Q4.csv')


<class 'pandas.DataFrame'>
RangeIndex: 2260701 entries, 0 to 2260700
Columns: 151 entries, id to settlement_term
dtypes: float64(113), object(1), str(37)
memory usage: 3.0+ GB


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,hardship_payoff_balance_amount,hardship_last_payment_amount,disbursement_method,debt_settlement_flag,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
0,68407277,NaN,3600.0,3600.0,3600.0,36 months,13.99,123.03,C,C4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
1,68355089,NaN,24700.0,24700.0,24700.0,36 months,11.99,820.28,C,C1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
2,68341763,NaN,20000.0,20000.0,20000.0,60 months,10.78,432.66,B,B4,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
3,66310712,NaN,35000.0,35000.0,35000.0,60 months,14.85,829.90,C,C5,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN
4,68476807,NaN,10400.0,10400.0,10400.0,60 months,22.45,289.91,F,F1,...,NaN,NaN,Cash,N,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
df.duplicated().sum()
df.dtypes

id                        object
member_id                float64
loan_amnt                float64
funded_amnt              float64
funded_amnt_inv          float64
                          ...   
settlement_status            str
settlement_date              str
settlement_amount        float64
settlement_percentage    float64
settlement_term          float64
Length: 151, dtype: object

In [4]:
cat_col = [col for col in df.columns if df[col].dtype == 'object']
num_col = [col for col in df.columns if df[col].dtype != 'object']

print('Categorical columns:', cat_col)
print('Numerical columns:', num_col)

Categorical columns: ['id']
Numerical columns: ['member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate', 'installment', 'grade', 'sub_grade', 'emp_title', 'emp_length', 'home_ownership', 'annual_inc', 'verification_status', 'issue_d', 'loan_status', 'pymnt_plan', 'url', 'desc', 'purpose', 'title', 'zip_code', 'addr_state', 'dti', 'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'initial_list_status', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low', 'collections_12_mths_ex_med', 'mths_since_last_major_derog', 'policy_code', 'application_type', 'annual_inc_joint', 'dti_joint', 'veri

In [5]:
df[cat_col].nunique()


id    2260701
dtype: int64

In [6]:
round((df.isnull().sum() / df.shape[0]) * 100, 2) # checking to see the percentage of missing values. if 80%, we usually remove the whole column

id                         0.00
member_id                100.00
loan_amnt                  0.00
funded_amnt                0.00
funded_amnt_inv            0.00
                          ...  
settlement_status         98.49
settlement_date           98.49
settlement_amount         98.49
settlement_percentage     98.49
settlement_term           98.49
Length: 151, dtype: float64

In [7]:
missing = (df.isnull().sum() / df.shape[0]) * 100  # identifying the columns with 80%+ missing values
cols_to_drop = missing[missing > 50].index

In [8]:
cols_to_drop = missing[missing > 50].index # counting the number of columns we are dropping
len(cols_to_drop)

44

In [9]:
df = df.drop(columns=cols_to_drop) # dropping the columns

In [10]:
(df.isnull().sum() / df.shape[0] * 100).sort_values(ascending=False) # checking to see if those actually have been dropped

il_util                 47.281042
mths_since_rcnt_il      40.251099
all_util                38.323555
open_acc_6m             38.313912
total_cu_tl             38.313912
                          ...    
application_type         0.001460
disbursement_method      0.001460
hardship_flag            0.001460
debt_settlement_flag     0.001460
id                       0.000000
Length: 107, dtype: float64

In [11]:


def detect_outliers_iqr(df, column):   # calculating outliers mathematically 
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    return ((df[column] < lower) | (df[column] > upper)).sum()

In [12]:
# Refresh column lists after dropping columns
cat_col = [col for col in df.columns if df[col].dtype == 'object']
num_col = [col for col in df.columns if df[col].dtype != 'object']

print(f"Updated numerical columns: {len(num_col)}")
print(f"Updated categorical columns: {len(cat_col)}")

Updated numerical columns: 106
Updated categorical columns: 1


In [13]:
outliers = {}

for col in num_col:
    # Only process truly numeric columns (int/float)
    if pd.api.types.is_numeric_dtype(df[col]):
        outliers[col] = detect_outliers_iqr(df, col)

outliers

{'loan_amnt': np.int64(35215),
 'funded_amnt': np.int64(35215),
 'funded_amnt_inv': np.int64(35212),
 'int_rate': np.int64(41099),
 'installment': np.int64(66312),
 'annual_inc': np.int64(110041),
 'dti': np.int64(21580),
 'delinq_2yrs': np.int64(421531),
 'fico_range_low': np.int64(74846),
 'fico_range_high': np.int64(74846),
 'inq_last_6mths': np.int64(94314),
 'open_acc': np.int64(84754),
 'pub_rec': np.int64(357881),
 'revol_bal': np.int64(137095),
 'revol_util': np.int64(114),
 'total_acc': np.int64(39411),
 'out_prncp': np.int64(212242),
 'out_prncp_inv': np.int64(212305),
 'total_pymnt': np.int64(83848),
 'total_pymnt_inv': np.int64(84034),
 'total_rec_prncp': np.int64(64014),
 'total_rec_int': np.int64(163420),
 'total_rec_late_fee': np.int64(87155),
 'recoveries': np.int64(185432),
 'collection_recovery_fee': np.int64(177013),
 'last_pymnt_amnt': np.int64(313415),
 'last_fico_range_high': np.int64(98414),
 'last_fico_range_low': np.int64(98414),
 'collections_12_mths_ex_med': 

In [14]:
outlier_percent = {}

for col in num_col:
    # Only process truly numeric columns (int/float)
    if pd.api.types.is_numeric_dtype(df[col]):
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        
        outlier_percent[col] = ((df[col] < lower) | (df[col] > upper)).mean() * 100

# Display outlier percentages sorted by highest to lowest
pd.Series(outlier_percent).sort_values(ascending=False).round(2)

num_accts_ever_120_pd    22.25
delinq_2yrs              18.65
pub_rec                  15.83
tot_coll_amt             14.79
last_pymnt_amnt          13.86
                         ...  
revol_util                0.01
bc_util                   0.00
mths_since_recent_inq     0.00
policy_code               0.00
percent_bc_gt_75          0.00
Length: 83, dtype: float64

In [15]:
# recompute numeric column list in case it changed earlier
num_col = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
for col in num_col:
    df[col] = df[col].fillna(df[col].median())

In [16]:
# recompute categorical columns list after dropping and numeric fill
cat_col = [col for col in df.columns if df[col].dtype == 'object']
for col in cat_col:
    if df[col].isnull().any():
        modes = df[col].mode()
        fill_value = modes[0] if len(modes) > 0 else 'Unknown'
        df[col] = df[col].fillna(fill_value)

In [17]:
df.isnull().sum().sum()

np.int64(340490)

In [18]:
missing_cols = df.isnull().sum()
missing_cols = missing_cols[missing_cols > 0].sort_values(ascending=False)
missing_cols

emp_title               167002
emp_length              146940
title                    23359
last_pymnt_d              2460
last_credit_pull_d         105
earliest_cr_line            62
zip_code                    34
verification_status         33
home_ownership              33
grade                       33
sub_grade                   33
term                        33
url                         33
issue_d                     33
loan_status                 33
addr_state                  33
purpose                     33
pymnt_plan                  33
initial_list_status         33
application_type            33
hardship_flag               33
disbursement_method         33
debt_settlement_flag        33
dtype: int64

In [19]:
df = df.drop(columns=['emp_title']) # dealing with the remaining missing values in each column
df['emp_length'] = df['emp_length'].fillna('Unknown')
df = df.dropna()
df.isnull().sum().sum()

np.int64(0)

In [20]:
# Encoding ordinal variables Encode grade (A < B < C < ... < G)
df['grade'] = df['grade'].astype(str).str.strip().str.upper()  # ← new cleaning line

grade_order = ['A','B','C','D','E','F','G']
df['grade'] = pd.Categorical(df['grade'], categories=grade_order, ordered=True)
df['grade'] = df['grade'].cat.codes

# Verify
print((df['grade'] == -1).sum())
print(df['grade'].unique())

0
[2 5 1 0 4 3 6]


In [21]:
print(df['emp_length'].unique())
print(repr(df['emp_length'].iloc[0]))

<ArrowStringArray>
['10+ years',   '3 years',   '4 years',   '6 years',    '1 year',   '7 years',
   '8 years',   '5 years',   '2 years',   '9 years',  '< 1 year',   'Unknown']
Length: 12, dtype: str
'10+ years'


In [22]:
# Clean first
df['emp_length'] = df['emp_length'].astype(str).str.strip()

# Then encode
emp_order = [
'< 1 year','1 year','2 years','3 years','4 years','5 years',
'6 years','7 years','8 years','9 years','10+ years','Unknown'
]
df['emp_length'] = pd.Categorical(df['emp_length'], categories=emp_order, ordered=True)
df['emp_length'] = df['emp_length'].cat.codes

# Verify
print((df['emp_length'] == -1).sum())
print(df['emp_length'].unique())

0
[10  3  4  6  1  7  8  5  2  9  0 11]


In [23]:
# Diagnose first
print("Before:", df['term'].unique())
print("Sample value:", repr(df['term'].iloc[0]))

# Extract and convert
df['term'] = df['term'].str.extract(r'(\d+)').astype(int)

# Verify
print("After:", df['term'].unique())
print("Dtype:", df['term'].dtype)

Before: <ArrowStringArray>
[' 36 months', ' 60 months']
Length: 2, dtype: str
Sample value: ' 36 months'
After: [36 60]
Dtype: int64


In [24]:
# Diagnose first
print("Before issue_d:", df['issue_d'].unique()[:5])
print("Before earliest_cr_line:", df['earliest_cr_line'].unique()[:5])

# Convert to datetime
df['issue_d'] = pd.to_datetime(df['issue_d'])
df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'])

# Verify
print("After issue_d dtype:", df['issue_d'].dtype)
print("After earliest_cr_line dtype:", df['earliest_cr_line'].dtype)
print("Sample issue_d:", df['issue_d'].iloc[0])
print("Sample earliest_cr_line:", df['earliest_cr_line'].iloc[0])

Before issue_d: <ArrowStringArray>
['Dec-2015', 'Nov-2015', 'Oct-2015', 'Sep-2015', 'Aug-2015']
Length: 5, dtype: str
Before earliest_cr_line: <ArrowStringArray>
['Aug-2003', 'Dec-1999', 'Sep-2008', 'Jun-1998', 'Oct-1987']
Length: 5, dtype: str


C:\Users\alahd\AppData\Local\Temp\ipykernel_17360\1894151616.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['issue_d'] = pd.to_datetime(df['issue_d'])
C:\Users\alahd\AppData\Local\Temp\ipykernel_17360\1894151616.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['earliest_cr_line'] = pd.to_datetime(df['earliest_cr_line'])


After issue_d dtype: datetime64[us]
After earliest_cr_line dtype: datetime64[us]
Sample issue_d: 2015-12-01 00:00:00
Sample earliest_cr_line: 2003-08-01 00:00:00


In [25]:
# Diagnose first
print("issue_d dtype:", df['issue_d'].dtype)
print("earliest_cr_line dtype:", df['earliest_cr_line'].dtype)

# Create feature
df['credit_history_years'] = df['issue_d'].dt.year - df['earliest_cr_line'].dt.year

# Verify
print("Sample values:", df['credit_history_years'].head())
print("Min:", df['credit_history_years'].min())
print("Max:", df['credit_history_years'].max())
print("Negative values:", (df['credit_history_years'] < 0).sum())
print("Nulls:", df['credit_history_years'].isnull().sum())

issue_d dtype: datetime64[us]
earliest_cr_line dtype: datetime64[us]
Sample values: 0    12
1    16
3     7
4    17
5    28
Name: credit_history_years, dtype: int32
Min: 0
Max: 83
Negative values: 0
Nulls: 0


In [26]:
# Drop columns
df = df.drop(columns=['issue_d','earliest_cr_line'])

# Verify
print("issue_d still exists:", 'issue_d' in df.columns)
print("earliest_cr_line still exists:", 'earliest_cr_line' in df.columns)
print("credit_history_years still exists:", 'credit_history_years' in df.columns)


issue_d still exists: False
earliest_cr_line still exists: False
credit_history_years still exists: True


In [27]:
# Diagnose first


# One-hot encode
nom_cols = [
'home_ownership',
'verification_status',
'purpose',
'initial_list_status',
'application_type',
'disbursement_method',
'sub_grade'
]

for col in nom_cols:
    print(f"\n{col}: {df[col].unique()}")


existing_cols = [col for col in nom_cols if col in df.columns]
df = pd.get_dummies(df, columns=existing_cols, drop_first=True)

# Verify
print("Shape after encoding:", df.shape)
print("Any original nom_cols still exist:", any(col in df.columns for col in nom_cols))
print("New columns created:", [col for col in df.columns if any(col.startswith(n) for n in nom_cols)])


home_ownership: <ArrowStringArray>
['MORTGAGE', 'RENT', 'OWN', 'ANY', 'NONE', 'OTHER']
Length: 6, dtype: str

verification_status: <ArrowStringArray>
['Not Verified', 'Source Verified', 'Verified']
Length: 3, dtype: str

purpose: <ArrowStringArray>
['debt_consolidation',     'small_business',     'major_purchase',
        'credit_card',   'home_improvement',              'other',
              'house',           'vacation',                'car',
            'medical',             'moving',   'renewable_energy',
            'wedding',        'educational']
Length: 14, dtype: str

initial_list_status: <ArrowStringArray>
['w', 'f']
Length: 2, dtype: str

application_type: <ArrowStringArray>
['Individual', 'Joint App']
Length: 2, dtype: str

disbursement_method: <ArrowStringArray>
['Cash', 'DirectPay']
Length: 2, dtype: str

sub_grade: <ArrowStringArray>
['C4', 'C1', 'C5', 'F1', 'C3', 'B2', 'B1', 'A2', 'B5', 'C2', 'E2', 'A4', 'E3',
 'A1', 'D4', 'B4', 'D1', 'B3', 'E4', 'D3', 'D2', 'D5', 'A

In [28]:
# Diagnose first
print("Before:", df['pymnt_plan'].unique())
print("Value counts:\n", df['pymnt_plan'].value_counts())

# Encode
df['pymnt_plan'] = df['pymnt_plan'].map({'n':0, 'y':1})

# Verify
print("After:", df['pymnt_plan'].unique())
print("Nulls:", df['pymnt_plan'].isnull().sum())
print("Dtype:", df['pymnt_plan'].dtype)

Before: <ArrowStringArray>
['n', 'y']
Length: 2, dtype: str
Value counts:
 pymnt_plan
n    2234241
y        612
Name: count, dtype: int64
After: [0 1]
Nulls: 0
Dtype: int64


In [29]:
# Diagnose first
print("Before:", df['loan_status'].unique())
print("Value counts:\n", df['loan_status'].value_counts())

# Encode target
df['loan_status'] = df['loan_status'].map({
    'Fully Paid': 0,
    'Charged Off': 1
})

# Verify
print("After:", df['loan_status'].unique())
print("Nulls:", df['loan_status'].isnull().sum())
print("Dtype:", df['loan_status'].dtype)
print("Class distribution:\n", df['loan_status'].value_counts(normalize=True).round(3))

Before: <ArrowStringArray>
[                                         'Fully Paid',
                                             'Current',
                                         'Charged Off',
                                     'In Grace Period',
                                  'Late (31-120 days)',
                                   'Late (16-30 days)',
                                             'Default',
  'Does not meet the credit policy. Status:Fully Paid',
 'Does not meet the credit policy. Status:Charged Off']
Length: 9, dtype: str
Value counts:
 loan_status
Fully Paid                                             1064878
Current                                                 872091
Charged Off                                             261442
Late (31-120 days)                                       21083
In Grace Period                                           8330
Late (16-30 days)                                         4284
Does not meet the credit policy. Status:Fu

In [30]:
df.select_dtypes(include=['object']).columns

C:\Users\alahd\AppData\Local\Temp\ipykernel_17360\549554427.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.select_dtypes(include=['object']).columns


Index(['id', 'url', 'title', 'zip_code', 'addr_state', 'last_pymnt_d',
       'last_credit_pull_d', 'hardship_flag', 'debt_settlement_flag'],
      dtype='str')

In [31]:
# Stopped at Encodingc... Encoding done successfully but needs attention!!! review to understand

In [32]:
                                        ## CHECKING OUR DATA IS CLEAN AND GETTING READY TO PROCEED WITH TRAINING ##

In [33]:

cols_to_drop = ['id', 'url', 'title', 'zip_code', 
                'last_pymnt_d', 'last_credit_pull_d', 
                'hardship_flag', 'debt_settlement_flag']  # dropping values that should have dropped earlier because they are not neeeded

bool_cols = df.select_dtypes(include='bool').columns # converting bool to int
df[bool_cols] = df[bool_cols].astype(int)

df = df.drop(columns=cols_to_drop, errors='ignore')
print(df.shape)
df.dtypes.value_counts()


(2234853, 147)


float64    84
int64      59
int8        2
str         1
int32       1
Name: count, dtype: int64

In [34]:

df = df.dropna(subset=['loan_status']) # fixing loan_status column has NaN values, just dropping them
df['loan_status'] = df['loan_status'].astype(int)
df.isnull().sum().sum()



np.int64(0)

In [35]:
df['loan_status'].value_counts(normalize=True).round(3)

loan_status
0    0.803
1    0.197
Name: proportion, dtype: float64

In [36]:
df.select_dtypes(include='object').columns.tolist()


C:\Users\alahd\AppData\Local\Temp\ipykernel_17360\3644157995.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.select_dtypes(include='object').columns.tolist()


['addr_state']

In [37]:
np.isinf(df.select_dtypes(include='number')).sum().sum()

np.int64(0)

In [38]:

leakage_cols = [
    # Payment activity — only known after loan is active                          # dropping values that will cause leakage 
    'out_prncp', 'out_prncp_inv',
    'total_pymnt', 'total_pymnt_inv',
    'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee',
    'recoveries', 'collection_recovery_fee',
    'last_pymnt_amnt',
    
    # FICO scores pulled after loan issued
    'last_fico_range_high', 'last_fico_range_low',
    
    # Delinquency during loan
    'acc_now_delinq', 'delinq_amnt',
    'num_tl_120dpd_2m', 'num_tl_30dpd',
    'num_tl_90g_dpd_24m', 'chargeoff_within_12_mths',
]

df = df.drop(columns=leakage_cols, errors='ignore')
df = df.drop(columns=['policy_code'], errors='ignore')
df = df.drop(columns=['funded_amnt', 'funded_amnt_inv'], errors='ignore')
df.columns.tolist()

['loan_amnt',
 'term',
 'int_rate',
 'installment',
 'grade',
 'emp_length',
 'annual_inc',
 'loan_status',
 'pymnt_plan',
 'addr_state',
 'dti',
 'delinq_2yrs',
 'fico_range_low',
 'fico_range_high',
 'inq_last_6mths',
 'open_acc',
 'pub_rec',
 'revol_bal',
 'revol_util',
 'total_acc',
 'collections_12_mths_ex_med',
 'tot_coll_amt',
 'tot_cur_bal',
 'open_acc_6m',
 'open_act_il',
 'open_il_12m',
 'open_il_24m',
 'mths_since_rcnt_il',
 'total_bal_il',
 'il_util',
 'open_rv_12m',
 'open_rv_24m',
 'max_bal_bc',
 'all_util',
 'total_rev_hi_lim',
 'inq_fi',
 'total_cu_tl',
 'inq_last_12m',
 'acc_open_past_24mths',
 'avg_cur_bal',
 'bc_open_to_buy',
 'bc_util',
 'mo_sin_old_il_acct',
 'mo_sin_old_rev_tl_op',
 'mo_sin_rcnt_rev_tl_op',
 'mo_sin_rcnt_tl',
 'mort_acc',
 'mths_since_recent_bc',
 'mths_since_recent_inq',
 'num_accts_ever_120_pd',
 'num_actv_bc_tl',
 'num_actv_rev_tl',
 'num_bc_sats',
 'num_bc_tl',
 'num_il_tl',
 'num_op_rev_tl',
 'num_rev_accts',
 'num_rev_tl_bal_gt_0',
 'num_sat

In [39]:
df.describe().T[['mean','min','max','std']]

,mean,min,max,std
loan_amnt,14417.840133,500.00,40000.00,8714.964473
term,41.786531,36.00,60.00,10.266104
int_rate,13.231249,5.31,30.99,4.760632
installment,437.965627,4.93,1719.83,261.396441
grade,1.742996,0.00,6.00,1.294035
...,...,...,...,...
sub_grade_G1,0.002207,0.00,1.00,0.046925
sub_grade_G2,0.001570,0.00,1.00,0.039589
sub_grade_G3,0.001184,0.00,1.00,0.034385
sub_grade_G4,0.000932,0.00,1.00,0.030513


In [40]:
df = pd.get_dummies(df, columns=['addr_state'], drop_first=True)
df.dtypes.value_counts()


float64    62
int64      60
bool       50
int8        2
int32       1
Name: count, dtype: int64

In [41]:
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)
df.dtypes.value_counts()

int64      110
float64     62
int8         2
int32        1
Name: count, dtype: int64

In [42]:
                                                     ## START TRAINIGN/SPLITING ##

In [43]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['loan_status'])
y = df['loan_status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train distribution:\n", y_train.value_counts(normalize=True).round(3))
print("y_test distribution:\n", y_test.value_counts(normalize=True).round(3))

X_train shape: (1061056, 174)
X_test shape: (265264, 174)
y_train distribution:
 loan_status
0    0.803
1    0.197
Name: proportion, dtype: float64
y_test distribution:
 loan_status
0    0.803
1    0.197
Name: proportion, dtype: float64


In [44]:
##################################### TRAINING / SPLITTING DONE SUCCESSFULLY ###############################

In [45]:
################################### START TRAINNING #################################

In [46]:
import sys
!{sys.executable} -m pip install lightgbm

In [47]:
import lightgbm as lgb
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

# Define model
model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    scale_pos_weight=0.803/0.197,  # handles class imbalance
    random_state=42,
    n_jobs=-1
)

# Train
model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
)

# Predict
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Evaluate
# Evaluate
print("AUC-ROC:", round(roc_auc_score(y_test, y_pred_proba), 4))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 209154, number of negative: 851902
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.214974 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 8154
[LightGBM] [Info] Number of data points in the train set: 1061056, number of used features: 172
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.197119 -> initscore=-1.404401
[LightGBM] [Info] Start training from score -1.404401
AUC-ROC: 0.7387

Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.66      0.76    212976
           1       0.33      0.69      0.45     52288

    accuracy                           0.66    265264
   macro avg       0.61      0.67      0.60    265264
weighted avg       0.78     

In [48]:
############################################### TRAINING DONE SUCCESSFYLLY  ##########################################

In [49]:
############################################### Hyperparameter Tuning START HERE  ################

In [50]:
# import sys
# !{sys.executable} -m pip install optuna

In [51]:

# # SKIP - tuning did not improve baseline model
# # study = optuna.create_study(...)

# import optuna
# from sklearn.model_selection import cross_val_score, train_test_split

# optuna.logging.set_verbosity(optuna.logging.WARNING)

# # Use only 20% of training data for tuning
# X_tune, _, y_tune, _ = train_test_split(
#     X_train, y_train, train_size=0.2, random_state=42, stratify=y_train
# )

# def objective(trial):
#     params = {
#         'n_estimators': trial.suggest_int('n_estimators', 200, 1000),
#         'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
#         'num_leaves': trial.suggest_int('num_leaves', 20, 100),
#         'min_child_samples': trial.suggest_int('min_child_samples', 20, 100),
#         'subsample': trial.suggest_float('subsample', 0.6, 1.0),
#         'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
#         'scale_pos_weight': 0.803/0.197,
#         'random_state': 42,
#         'n_jobs': -1
#     }
    
#     model = lgb.LGBMClassifier(**params)
#     score = cross_val_score(model, X_tune, y_tune,
#                            cv=3, scoring='roc_auc').mean()
#     return score

# # Run optimization
# study = optuna.create_study(direction='maximize')
# study.optimize(objective, n_trials=20, show_progress_bar=True)

# print("Best AUC-ROC:", round(study.best_value, 4))
# print("Best parameters:", study.best_params)

In [52]:
print("Best AUC-ROC:", round(study.best_value, 4))


NameError: name 'study' is not defined

In [ ]:
######################################## Saving the Model ##################################

In [ ]:
import joblib

joblib.dump(model, 'loan_default_model.pkl')
print("Model saved successfully!")

Model saved successfully!


In [ ]:
# Test on a single real loan from test set
single_loan = X_test.iloc[0]
prediction = model.predict([single_loan])
probability = model.predict_proba([single_loan])[:, 1]

print(prediction[0])
print("Prediction:", "Default" if prediction[0] == 1 else "Fully Paid")
print("Default Probability:", round(probability[0], 4))
print("Actual Label:", "Default" if y_test.iloc[0] == 1 else "Fully Paid")

[0]
Prediction: Fully Paid
Default Probability: 0.2612
Actual Label: Fully Paid


c:\Users\alahd\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\alahd\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
